# Phase 6 - Model Improvement

This is where we take the baseline Linear Regression model from Phase 4?5 and improve it in a systematic way.

The key principle is:

> Diagnose -> change one thing -> retrain -> evaluate -> compare.

The full workflow is:

```text
1. Data Understanding & Cleaning
   ?
2. EDA
   ?
3. Feature Engineering
   ?
4. Baseline Linear Regression
   ?
5. Model Evaluation
   ?
? 6. Model Improvement
   ?
7. Prediction API
   ?
8. Dashboard
```


## 6.1 First understand what we're improving

Your baseline model was trained as:

```python
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

And we measured the main metrics with MAE, RMSE and R?.

The goal of Phase 6 is not to add complexity randomly. The goal is to diagnose the weakness, make one focused change, retrain, and compare the results.


In [1]:
from pathlib import Path

import joblib
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
MODEL_PATH = PROJECT_DIR / 'models' / 'sales_linear_regression.pkl'

X_train = pd.read_csv(DATA_DIR / 'X_train.csv', index_col=0)
y_train = pd.read_csv(DATA_DIR / 'y_train.csv', index_col=0)['Sales']
X_test = pd.read_csv(DATA_DIR / 'X_test.csv', index_col=0)
y_test = pd.read_csv(DATA_DIR / 'y_test.csv', index_col=0)['Sales']

baseline_model = joblib.load(MODEL_PATH)

print(f'Baseline model loaded from: {MODEL_PATH}')
print(f'Train rows: {len(X_train):,}; test rows: {len(X_test):,}')
print(f'Baseline feature columns: {list(X_train.columns)}')


Baseline model loaded from: E:\NIAI\Projects\Retail Sales Prediction\models\sales_linear_regression.pkl
Train rows: 675,513; test rows: 168,879
Baseline feature columns: ['Store', 'Customers', 'Promo', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend']


## 6.2 Our Phase 6 improvement strategy

We will improve the model in the following order:

```text
Improvement 1
Better categorical handling
        ?
Improvement 2
Time-aware train/test split
        ?
Improvement 3
Better date/time features
        ?
Improvement 4
Feature transformations
        ?
Improvement 5
Feature selection
        ?
Improvement 6
Regularized Linear Regression
        ?
Final comparison
```

There is one particularly important issue in the baseline model: `Store` should not be treated as a raw continuous number. The difference between `Store 10` and `Store 20` is not a meaningful numeric ratio. This is why we use categorical encoding.


## 6.3 Improvement #1 ? Proper categorical encoding

Instead of treating `Store`, `Promo`, and `SchoolHoliday` as regular numeric values, we convert them into separate indicator columns so the model understands them as categories.

This is done using a `ColumnTransformer` and `OneHotEncoder`.


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    'Customers',
    'Year',
    'Month',
    'Day',
    'WeekOfYear',
    'IsWeekend'
]

categorical_features = [
    'Store',
    'DayOfWeek',
    'Promo',
    'SchoolHoliday',
    'StateHoliday'
]

numeric_features = [feature for feature in numeric_features if feature in X_train.columns]
categorical_features = [feature for feature in categorical_features if feature in X_train.columns]

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model_v2 = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

model_v2


Numeric features: ['Customers', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend']
Categorical features: ['Store', 'Promo', 'SchoolHoliday']


In [3]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_test)

mae_v2 = mean_absolute_error(y_test, y_pred_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_test, y_pred_v2))
r2_v2 = r2_score(y_test, y_pred_v2)

print('Model V2')
print('MAE :', mae_v2)
print('RMSE:', rmse_v2)
print('R?  :', r2_v2)


Model V2
MAE : 464.40174716183935
RMSE: 682.4062559169372
R?  : 0.9515067004448071


## 6.8 Compare Model V1 vs V2

Now we compare the baseline model against the encoded version. Lower MAE/RMSE is better; higher R? is better.


In [4]:
import pandas as pd

baseline_pred = baseline_model.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

comparison = pd.DataFrame({
    'Model': ['Baseline Linear Regression', 'Encoded Linear Regression'],
    'MAE': [baseline_mae, mae_v2],
    'RMSE': [baseline_rmse, rmse_v2],
    'R2': [baseline_r2, r2_v2]
})

display(comparison)

mae_improvement = ((baseline_mae - mae_v2) / baseline_mae) * 100
rmse_improvement = ((baseline_rmse - rmse_v2) / baseline_rmse) * 100
r2_improvement = ((r2_v2 - baseline_r2) / abs(baseline_r2)) * 100

print(f'MAE improvement: {mae_improvement:.2f}%')
print(f'RMSE improvement: {rmse_improvement:.2f}%')
print(f'R? improvement: {r2_improvement:.2f}%')


                        Model          MAE         RMSE        R2
0  Baseline Linear Regression  1146.044504  1602.209864  0.732678
1   Encoded Linear Regression   464.401747   682.406256  0.951507
MAE improvement: 59.48%
RMSE improvement: 57.41%
R? improvement: 29.87%


## 6.10 Improvement #2 ? Fix the train/test strategy

A random split can look artificially good because it mixes past and future rows. A more realistic setup is to sort by time and use the earliest rows for training and the latest rows for testing.

This is closer to the business question: can the model predict future sales?


In [5]:
df_model = pd.read_csv(DATA_DIR / 'df_model.csv')
df_model['Date'] = pd.to_datetime(df_model['Date'])
df_model = df_model.sort_values('Date').copy()

split_index = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_index].copy()
test_df = df_model.iloc[split_index:].copy()

features = [
    'Store', 'DayOfWeek', 'Customers', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend'
]

X_train_time = train_df[features]
y_train_time = train_df['Sales']
X_test_time = test_df[features]
y_test_time = test_df['Sales']

num_time = [f for f in numeric_features if f in X_train_time.columns]
cat_time = [f for f in categorical_features if f in X_train_time.columns]
preprocessor_time = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_time),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_time)
    ]
)

model_v3 = Pipeline(
    steps=[
        ('preprocessor', preprocessor_time),
        ('model', LinearRegression())
    ]
)

model_v3.fit(X_train_time, y_train_time)
y_pred_v3 = model_v3.predict(X_test_time)

mae_v3 = mean_absolute_error(y_test_time, y_pred_v3)
rmse_v3 = np.sqrt(mean_squared_error(y_test_time, y_pred_v3))
r2_v3 = r2_score(y_test_time, y_pred_v3)

print('Time-based model')
print('MAE :', mae_v3)
print('RMSE:', rmse_v3)
print('R²  :', r2_v3)


Time-based model
MAE : 470.73872476979227
RMSE: 710.5206688960952
R²  : 0.9462026018433017


## 6.14 Improvement #3 ? Better time features

`Month` and `DayOfWeek` are cyclical. December and January are near each other on a calendar, but numerically they are far apart. Sine/cosine encoding makes this relationship explicit.


In [6]:
df_model['MonthSin'] = np.sin(2 * np.pi * df_model['Month'] / 12)
df_model['MonthCos'] = np.cos(2 * np.pi * df_model['Month'] / 12)
df_model['DayOfWeekSin'] = np.sin(2 * np.pi * df_model['DayOfWeek'] / 7)
df_model['DayOfWeekCos'] = np.cos(2 * np.pi * df_model['DayOfWeek'] / 7)

df_model[['Month', 'MonthSin', 'MonthCos', 'DayOfWeek', 'DayOfWeekSin', 'DayOfWeekCos']].head()


## 6.17 Improvement #4 ? Consider logarithmic transformation

Retail sales are often skewed. If residual plots or the sales distribution show a strong right tail, a log transformation can sometimes help. But do not do this automatically; the evaluation must transform the target back before comparing values in the original sales scale.


## 6.18 Improvement #5 ? Feature selection

More features do not always mean a better model. We should inspect the relationship between candidate features and the target before assuming that every feature helps.

```python
corr = df_model[[
    'Sales', 'Customers', 'Promo', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend'
]].corr()['Sales'].sort_values(ascending=False)
print(corr)
```

Correlation helps us explore relationships, but multivariable model performance remains the real judge.


## 6.19 Improvement #6 ? Regularization

When one-hot encoding creates many columns, we can stabilize the model with regularized linear regression. Ridge is a good first test because it adds a penalty for large coefficients without removing features completely.


In [7]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', Ridge(alpha=1.0))
    ]
)

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print('Ridge Regression')
print('MAE :', ridge_mae)
print('RMSE:', ridge_rmse)
print('R?  :', ridge_r2)


Ridge Regression
MAE : 1149.4084157879215
RMSE: 1607.2117740039625
R?  : 0.731006400374729


In [8]:
results = pd.DataFrame({
    'Model': [
        'Baseline Linear Regression',
        'Encoded Linear Regression',
        'Time-based Linear Regression',
        'Ridge Regression'
    ],
    'MAE': [baseline_mae, mae_v2, mae_v3, ridge_mae],
    'RMSE': [baseline_rmse, rmse_v2, rmse_v3, ridge_rmse],
    'R2': [baseline_r2, r2_v2, r2_v3, ridge_r2]
})

display(results.sort_values('MAE'))


                          Model          MAE         RMSE        R2
1     Encoded Linear Regression   464.401747   682.406256  0.951507
2  Time-based Linear Regression   470.738725   710.520669  0.946203
0    Baseline Linear Regression  1146.044504  1602.209864  0.732678
3              Ridge Regression  1149.408416  1607.211774  0.731006
